# GloVe, FastText, and Subword Embedding

Word2Vec trained one embedding **per word.**

* GloVe factorized co-occurrence matrix.
* FastText embedded the pieces.
* BPE bridged to transformers.

## Problem Definition

Word2Vec left open questions.

* Parallel. Matrix factorization with a thoughtfully chosen loss matches or beats Word2Vec, and costs less to train.
* Words never seen.
* Word-level vocabularies cap out around a million entries.

## Basic Concept

### GloVe (Global Vectors)

Build the word-word co-occurrence matrix $X$ where $X_{ij}$ is how often word $j$ appears in the context of word $i$. Train vectors such that

$$
v_i^{\top} v_j + b_i + b_j \approx \log X_{ij}
$$

#### What $v$ and $b$ are

- **$v$**: word **embedding vectors**
  - $v_i$: vector for word $i$
  - $v_j$: vector for word $j$
  - Dot product $v_i^{\top} v_j$: captures how related the two words are (co-occurrence / semantics)

- **$b$**: scalar **biases**
  - $b_i$: depends only on word $i$ (e.g. how frequent / how often it acts as a center word)
  - $b_j$: depends only on word $j$
  - Absorb terms that do **not** depend on the other word (such as $\log X_i$), so the dot product focuses on the pairwise relation

After training, downstream tasks usually keep $v$ (sometimes average word and context vectors). The biases $b$ are mostly training machinery.

#### Where the objective comes from (short)

GloVe starts from co-occurrence **ratios** $P(k\mid i)/P(k\mid j)$, which encode meaning better than raw counts. Under symmetry / decomposability constraints this collapses to a log-bilinear form; single-sided terms become biases, yielding $v_i^{\top} v_j + b_i + b_j \approx \log X_{ij}$.


### FastText
A word is the sum of its character n-grams plus the word itself. `where` becomes `<wh, whe, her, ere, re>, <where>`. The word vector is the sum of those component vectors. Train as Word2Vec. Benefit: unseen words (`whereupon`) compose from known n-grams.

### BPE (Byte-Pair Encoding)
Start with a vocabulary of individual bytes (or characters). Count every adjacent pair in the corpus. Merge the most frequent pair into a new token. Repeat for `k` iterations. Result: a vocabulary of `k + 256` tokens where frequent sequences (`ing`, `tion`, `the`) are single tokens and rare words are broken into familiar pieces. Every sentence tokenizes into something.


# Build your Own

## GloVe

In [ ]:
import numpy as np
from collections import Counter

from sympy.tensor import indexed

def build_coocurrence(docs, windows=5):
    pairs_counts = Counter()
    vocab = {}
    for doc in docs:
        for token in doc:
            if token not in vocab:
                vocab[token] = len(vocab)
    for doc in docs:
        indexed = [vocab[t] for t in doc]
        for i, center in enumerate(indexed):
            for j in range(max(0, 1- windows), min(len(indexed), i + windows + 1)):
                if i != j:
                    distance = abs(i - j)
                    pairs_counts[(center, indexed[j])] += 1.0 / distance

    return vocab, pairs_counts


build_coocurrence([["hello", "world"], ["hello", "there"]], windows=2)

def glove_train(vocab, pairs_counts, dim=16, epochs=100, lr=0.5, x_max=100, alpha=0.75, seed=0):
    n = len(vocab)
    rng = np.random.default_rng(seed)

    W = rng.uniform(0, 0.1, size = (n, dim))
    W_tilde = rng.uniform(0, 0.1, size = (n, dim))
    b = np.zeros(n)
    b_tilde = np.zeros(n)

    for epoch in range(epochs):
        for (i, j), x_ij in pairs_counts.items():
            weight = (x_ij / x_max) ** alpha if x_ij < x_max else 1.0
            diff = W[i] @ W_tilde[j] + b[i] + b_tilde[j] - np.log(x_ij)
            coef = weight * diff

            grad_W_i = coef * W_tilde[j]
            grad_W_tilde_j = coef * W[i]
            W[i] -= lr * grad_W_i
            W_tilde[j] -= lr * grad_W_tilde_j
            b[i] -= lr * coef
            b_tilde[j] -= lr * coef

    return W, b, W_tilde, b_tilde
    

({'hello': 0, 'world': 1, 'there': 2},
 Counter({(0, 1): 1.0, (1, 0): 1.0, (0, 2): 1.0, (2, 0): 1.0}))

## FastText

In [5]:
def char_ngrams(word, n_min=3, n_max=6):
    wrapped = f"<{word}>"
    grams = {wrapped}
    for n in range(n_min, n_max + 1):
        for i in range(len(wrapped) - n + 1):
            grams.add(wrapped[i:i+n])
    return grams

char_ngrams("where")

def fasttext_vector(word, ngram_table):
    grams = char_ngrams(word)
    vecs = [ngram_table[g] for g in grams if g in ngram_table]
    if not vecs:
        return None
    return np.sum(vecs, axis=0)


## BPE: Learned subword vocabulary

In [ ]:
def learn_bpe(corpus, k_merges):
    vocab = Counter()
    for word, freq in corpus.items():
        tokens = tuple(word) + ("</w>",)
        vocab[tokens] += freq

    merges = []
    for _ in range(k_merges):
        pair_freq = Counter()
        for tokens, freq in vocab.items():
            for a, b in zip(tokens, tokens[1:]):
                pair_freq[(a, b)] += freq
        if not pair_freq:
            break

        best = pair_freq.most_common(1)[0][0]
        merges.append(best)

        new_vocab = Counter()
        for tokens, freq in vocab.items():
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) == best:
                    new_tokens.append(tokens[i] + tokens[i + 1])
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            new_vocab[tuple(new_tokens)] = freq
        vocab = new_vocab
    return merges


corpus = Counter({"low": 5, "lower": 2, "newest": 6, "widest": 3})
merges = learn_bpe(corpus, k_merges=10)
# Combination of merges and vocab
print(merges)

def apply_bpe(text, merges):
    tokens = list(text) + ["</w>"]
    for a, b in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i + 1 < len(tokens) and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens

apply_bpe("lowest", merges)

[('e', 's'),
 ('es', 't'),
 ('est', '</w>'),
 ('l', 'o'),
 ('lo', 'w'),
 ('n', 'e'),
 ('ne', 'w'),
 ('new', 'est</w>'),
 ('low', '</w>'),
 ('w', 'i')]